# CBT-18.95M 1D-RoPE Twin: Position-Encoding Ablation (Dual NVIDIA T4)

Trains the CBT-18.95M architecture from scratch with standard 1D-RoPE (positions 0..T-1 over the full head_dim) instead of
decoupled 2D-RoPE. Parameters, data generators, seeds, batch sizes, learning rates and step counts match the 2D lineage.
Position encoding is the only variable.

| Stage | 2D lineage | 1D twin (this notebook) |
| :--- | :--- | :--- |
| Pretrain | 50,000 steps, LR 1e-3, 15k anneal, 20k+20k pool | identical |
| Search copy SFT (7,000 steps) + all-tools SFT (4,000 steps) | yes | not replayable (code superseded); skipped |
| V4 SFT | 12,000 steps, LR 5e-5, k=45 | identical |
| V5 SFT | 6.0 h time-annealed | `V5_STEPS` steps (set to the 2D run's final step), capped at 6.0 h |

Shift augmentation: a uniform position offset is an exact no-op under RoPE, so the model skips it for `rope_1d`.

**Sessions.** About 20 GPU-hours in total. Each session runs every stage that fits and stops cleanly before Kaggle's
12 h limit. To continue, start a new session, attach this notebook's previous output (Add Input > Notebook Output),
and run all cells again. The notebook imports earlier 1D checkpoints and resumes pretraining from its timeout checkpoint.
- Session 1: pretraining (resumable).
- Session 2: rest of pretraining (if any) + V4 SFT (needs 3.5 h).
- Session 3: V5 SFT (needs 6.5 h) + fast benchmarks.

In [ ]:
# Cell 1: Hardware check and session clock (Turing sm_75, FP16 only, TF32 off)
import time, torch
SESSION_T0 = time.time()
SESSION_HOURS = 11.6   # Kaggle GPU sessions stop at 12 h
def hours_left() -> float:
    return SESSION_HOURS - (time.time() - SESSION_T0) / 3600.0

print(f"[INFO] PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "[FAIL] No GPU. Enable 'GPU T4 x2' in the notebook accelerator settings."
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"[INFO] GPU {i}: {p.name} | {p.total_memory / 2**30:.1f} GB | sm_{p.major}{p.minor}")
if torch.cuda.device_count() < 2:
    print("[WARN] Fewer than 2 GPUs: the 2D lineage used 2. Throughput and DataParallel splits differ.")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

In [ ]:
# Cell 2: Workspace + import 1D checkpoints from earlier sessions (attached as input)
import os, sys, glob, shutil, zipfile

WORK = "/kaggle/working/cbt1d" if os.path.isdir("/kaggle/working") else os.path.abspath("cbt1d_workspace")
MARKER = "champion_model/scaled_t4/sft_search.py"
BUNDLE = "cbt_1d_ablation_bundle.zip"

def find_bundle_root() -> str:
    for base in [os.getcwd(), WORK]:
        if os.path.exists(os.path.join(base, MARKER)):
            return base
    for hit in sorted(glob.glob(f"/kaggle/input/**/{MARKER}", recursive=True)):
        return hit[: -len(MARKER)].rstrip("/")
    for z in glob.glob(f"/kaggle/input/**/{BUNDLE}", recursive=True) + glob.glob(BUNDLE):
        os.makedirs(WORK, exist_ok=True)
        zipfile.ZipFile(z).extractall(WORK)
        return WORK
    raise FileNotFoundError(f"[FAIL] Bundle not found. Attach {BUNDLE} as a dataset or upload it.")

root = find_bundle_root()
if os.path.abspath(root) != os.path.abspath(WORK):
    shutil.copytree(root, WORK, dirs_exist_ok=True)
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

CK = "checkpoints/cbt"
os.makedirs(CK, exist_ok=True)
for src in glob.glob("/kaggle/input/**/cbt_18m_1d_rope_*", recursive=True):
    dst = os.path.join(CK, os.path.basename(src))
    if os.path.isfile(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"[OK] Imported from earlier session: {os.path.basename(src)}")
print(f"[OK] Workspace: {WORK}")
print("[INFO] 1D checkpoints present:", sorted(os.path.basename(p) for p in glob.glob(f"{CK}/cbt_18m_1d_rope_*")) or "none")

In [ ]:
# Cell 3: Unit tests (architecture, 1D ablation path, KV decoder parity, masking, V5 decontamination)
import subprocess, sys
# discover -s loads tests/ by path: a pip-installed top-level "tests" package cannot shadow it
r = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_scaled_t4.py"],
                   capture_output=True, text=True)
print(r.stderr[-1500:])
assert r.returncode == 0, "[FAIL] Unit tests failed; do not launch training."
print("[OK] Unit tests passed.")

In [ ]:
# Cell 4: Stage paths, 2D-matched recipe, and a streaming runner
import glob, os, re, subprocess, sys, time
import torch

PRE = f"{CK}/cbt_18m_1d_rope_R3_k32_pretrain.pt"
V4 = f"{CK}/cbt_18m_1d_rope_R3_k45_sftv4.pt"
V5 = f"{CK}/cbt_18m_1d_rope_R3_k45_sftv5.pt"
V5_BEST = V5.replace(".pt", "_best.pt")
LOG_DIR = "/kaggle/working/logs_1d" if os.path.isdir("/kaggle/working") else "logs_1d"
os.makedirs(LOG_DIR, exist_ok=True)

# Final step of the 2D V5 run (champion_v3). Step-matching keeps the comparison fair if 1D is faster per step.
# 0 = time-matched instead (6.0 h, as the 2D run).
V5_STEPS = 23927

def run(cmd, log_name: str) -> int:
    env = os.environ.copy()
    env.update({"PYTHONUNBUFFERED": "1", "PYDEVD_DISABLE_FILE_VALIDATION": "1"})
    print("[INFO] " + " ".join(cmd), flush=True)
    t0 = time.time()
    with open(os.path.join(LOG_DIR, log_name), "a") as log:
        proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            log.write(line)
            log.flush()
            if "Warning" not in line and "warnings.warn" not in line:
                print(line, end="", flush=True)
        proc.wait()
    print(f"[INFO] Exit code {proc.returncode} after {(time.time() - t0) / 3600:.2f} h | {hours_left():.2f} h left in session")
    return proc.returncode

def latest_pretrain_resume():
    best, best_step = None, -1
    pats = [f"{CK}/cbt_18m_1d_rope_R3_k32_pretrain_*step_*.pt",
            "/kaggle/working/checkpoints/scaled_t4/cbt_18m_1d_rope_R3_k32_pretrain_*step_*.pt"]
    for p in sorted(set(sum((glob.glob(x) for x in pats), []))):
        m = re.search(r"step_(\d+)\.pt$", p)
        if m and int(m.group(1)) > best_step:
            best, best_step = p, int(m.group(1))
    return best, best_step

print(f"[INFO] Pretrain done: {os.path.exists(PRE)} | V4 done: {os.path.exists(V4)} | V5 done: {os.path.exists(V5)}")
print(f"[INFO] Latest pretrain resume point: {latest_pretrain_resume()}")

## Stage 1: Pretraining (50,000 steps, resumable)

Same command as the 2D run (`kaggle_scaled_champion.ipynb`, Cell 5) plus `--pos-emb rope_1d`.
On the time limit it writes `_timeout_step_N.pt` (model + optimizer + scheduler) and no final checkpoint.

In [ ]:
# Cell 5: Stage 1 pretraining
if os.path.exists(PRE) or os.path.exists(V4) or os.path.exists(V5):
    print("[OK] Pretraining already complete (pretrain or a later stage checkpoint is present).")
elif hours_left() < 0.75:
    print("[INFO] Not enough session time left; re-run in the next session.")
else:
    resume, resume_step = latest_pretrain_resume()
    cmd = [sys.executable, "-u", "champion_model/scaled_t4/train_and_eval.py",
           "--pos-emb", "rope_1d",
           "--steps", "50000", "--anneal-steps", "15000", "--batch-size", "32", "--lr", "1.0e-3",
           "--n-math", "20000", "--n-tools", "20000", "--log-interval", "50", "--save-interval", "2500",
           "--max-keep-ckpts", "2", "--max-hours", f"{hours_left() - 0.4:.2f}", "--save-ckpt", PRE]
    if resume:
        print(f"[INFO] Resuming from step {resume_step}: {resume}")
        cmd += ["--resume", resume]
    rc = run(cmd, "stage1_pretrain.log")
    assert rc == 0, "[FAIL] Pretraining crashed; see the log."
    print(f"[OK] Pretraining complete: {PRE}" if os.path.exists(PRE)
          else "[INFO] Session time used up. Start a new session with this output attached to resume.")

## Stage 2: V4 SFT (12,000 steps)

Same arguments as `train.sh ... sft` for the 2D champion. `--pos-emb rope_1d` aborts if the base checkpoint is not 1D.
It is not resumable, so it starts only with at least 3.5 h left.

In [ ]:
# Cell 6: Stage 2 V4 SFT
if os.path.exists(V4) or os.path.exists(V5):
    print("[OK] V4 SFT already complete (V4 or V5 checkpoint is present).")
elif not os.path.exists(PRE):
    print("[INFO] Waiting for pretraining.")
elif hours_left() < 3.5:
    print(f"[INFO] {hours_left():.2f} h left < 3.5 h needed; re-run in the next session.")
else:
    rc = run([sys.executable, "-u", "champion_model/scaled_t4/sft_search.py", "--pos-emb", "rope_1d",
              "--base-ckpt", PRE, "--steps", "12000", "--batch-size", "32", "--lr", "5.0e-5",
              "--k-rand-shift", "45", "--save-ckpt", V4, "--max-keep-ckpts", "2"], "stage2_sft_v4.log")
    assert rc == 0 and os.path.exists(V4), "[FAIL] V4 SFT failed; see the log."
    print(f"[OK] V4 SFT complete: {V4}")

## Stage 3: V5 SFT

Same arguments as `train.sh ... sft_v3` for the 2D champion_v3. Set `V5_STEPS` (Cell 4) to the 2D run's final step.
`_best.pt` is the highest tool held-out accuracy among checkpoints with the highest held-out arithmetic. That is the
same choice as the 2D rule whenever arithmetic reaches 100%.

In [ ]:
# Cell 7: Stage 3 V5 SFT
if os.path.exists(V5):
    print(f"[OK] V5 SFT already complete: {V5}")
elif not os.path.exists(V4):
    print("[INFO] Waiting for V4 SFT.")
elif hours_left() < 6.5:
    print(f"[INFO] {hours_left():.2f} h left < 6.5 h needed; re-run in the next session.")
else:
    steps = str(V5_STEPS) if V5_STEPS > 0 else "200000"
    if V5_STEPS <= 0:
        print("[WARN] V5_STEPS not set: time-matched (6.0 h) instead of step-matched.")
    rc = run([sys.executable, "-u", "champion_model/scaled_t4/sft_search.py", "--pos-emb", "rope_1d",
              "--curriculum", "v5", "--base-ckpt", V4, "--steps", steps, "--batch-size", "32", "--lr", "5.0e-5",
              "--k-rand-shift", "45", "--max-hours", "6.0", "--save-interval", "2000", "--probe-interval", "2000",
              "--save-ckpt", V5, "--max-keep-ckpts", "2"], "stage3_sft_v5.log")
    assert rc == 0 and os.path.exists(V5), "[FAIL] V5 SFT failed; see the log."
    print(f"[OK] V5 SFT complete: {V5}")

In [ ]:
# Cell 8: Fast benchmarks on every finished 1D stage (phrasing 26, OOD 18, unseen OOD 30, arithmetic 16-55d)
# Full sweeps (evaluate_champion --suite arithmetic, evaluate_random_16_to_100d to 100 digits) are slow; run them offline.
out_dir = "/kaggle/working/results_1d" if os.path.isdir("/kaggle/working") else "results_1d"
os.makedirs(out_dir, exist_ok=True)
stages = [("v5", V5_BEST if os.path.exists(V5_BEST) else V5), ("v4", V4), ("pretrain", PRE)]
for tag, ck in stages:
    if not os.path.exists(ck) or os.path.exists(f"{out_dir}/cbt1d_{tag}_unseen_ood.json"):
        continue
    if hours_left() < 0.5:
        print("[INFO] Not enough session time for benchmarks; run them offline or next session.")
        break
    c = torch.load(ck, map_location="cpu", weights_only=False)
    assert c["config"].pos_emb_type == "rope_1d", f"[FAIL] {ck} is not a 1D checkpoint"
    print(f"[INFO] {tag}: {ck} | step {c.get('step', c.get('steps'))} | tools {c.get('heldout_acc')} | math {c.get('math_heldout_acc')}")
    for suite in ("phrasing", "ood"):
        subprocess.run([sys.executable, "-u", "benchmarks/evaluate_champion.py", "--ckpt", ck, "--suite", suite,
                        "--out-json", f"{out_dir}/cbt1d_{tag}_{suite}.json"], check=True)
    subprocess.run([sys.executable, "-u", "benchmarks/evaluate_random_16_to_100d.py", "--ckpt", ck, "--max-digits", "55",
                    "--out-json", f"{out_dir}/cbt1d_{tag}_arith_16_55d.json"], check=True)
    subprocess.run([sys.executable, "-u", "benchmarks/evaluate_unseen_ood.py", "--ckpt", ck,
                    "--out-json", f"{out_dir}/cbt1d_{tag}_unseen_ood.json"], check=True)

In [ ]:
# Cell 9: Package finished checkpoints, logs and results for download
import zipfile
out = "/kaggle/working/cbt_1d_ablation_outputs.zip" if os.path.isdir("/kaggle/working") else "cbt_1d_ablation_outputs.zip"
files = [p for p in (PRE, V4, V5, V5_BEST, V5.replace(".pt", "_corpus.txt.gz")) if os.path.exists(p)]
if not os.path.exists(PRE) and latest_pretrain_resume()[0]:
    files.append(latest_pretrain_resume()[0])
files += sorted(glob.glob(f"{LOG_DIR}/*.log")) + sorted(glob.glob("/kaggle/working/results_1d/*.json"))
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, os.path.basename(f))
        print(f"[OK] {f} ({os.path.getsize(f) / 2**20:.1f} MB)")
print(f"[OK] Download: {out} ({os.path.getsize(out) / 2**20:.1f} MB)")